# Boundary-Aware PINNsformer — Kaggle Runbook

Run the whole project **one block at a time**, top to bottom. Each numbered section is independent: run its code cell(s), check the output, then move on. Your laptop (no GPU) handles §§0–6; the GPU blocks (§§7–8, 10) run here on Kaggle.

| Block | Needs | Where |
|---|---|---|
| §§0–4 setup + smoke | CPU | laptop or Kaggle |
| §5 E0 data pipeline | CPU | laptop or Kaggle |
| §6 E1 sweep | CPU | laptop or Kaggle |
| §7 E2 fits | **GPU T4** | Kaggle, ≤12 h/session, 30 h/week |
| §8 E3 ablations | **GPU T4** | Kaggle |
| §9 E4 fire validation | CPU | anywhere (needs E2 output + D4) |
| §10 E5 transfer | **GPU** | Kaggle |
| §11 figures | CPU | anywhere |

**Quota strategy:** one (city, season, pollutant) fit per session. To use both T4s, open this notebook twice (two sessions) and run *different* seasons/pollutants in each — never data-parallel, per PRD §9.

In [ ]:
# §0 — environment check. Run first.
import sys, shutil
print('python:', sys.version.split()[0])
try:
    import torch
    print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(), end='')
    if torch.cuda.is_available():
        print(f" | gpus: {torch.cuda.device_count()} x {torch.cuda.get_device_name(0)}")
    else:
        print('  (CPU-only session: §§0-6, 9, 11 only)')
except ImportError:
    print('torch NOT installed (will install below)')
for pkg in ('yaml', 'numpy', 'pandas', 'matplotlib', 'pytest'):
    try:
        __import__(pkg)
        print(f'{pkg:10s} ok')
    except ImportError:
        print(f'{pkg:10s} MISSING')
print('internet:', 'ok' if shutil.which('curl') else 'check manually')

## §1 — Install (selective: never downgrade Kaggle's torch)

Kaggle images already ship a CUDA torch. `pip install -r requirements.txt` would **downgrade** it, so install only what is missing.

In [ ]:
# §1 — run once per session.
import importlib, subprocess, sys

def ensure(import_name, pip_name=None):
    try:
        importlib.import_module(import_name)
        print(f'{import_name:12s} present, skipping')
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name or import_name], check=True)
        print(f'{import_name:12s} installed')

for mod, pip in [('yaml', 'pyyaml'), ('numpy', None), ('pandas', None),
                   ('scipy', None), ('matplotlib', None), ('pytest', None)]:
    ensure(mod, pip)
# Optional helpers (fetch-only; absence never breaks training):
for mod, pip in [('xarray', None), ('netCDF4', None), ('cdsapi', None), ('pyproj', None)]:
    try:
        ensure(mod, pip)
    except Exception as e:
        print(f'{mod:12s} optional, skipped ({e})')

## §2 — Get the code

Option A (recommended): clone from GitHub. Option B: upload the repo folder as a Kaggle dataset and attach it as input (then set `REPO` to its path).

In [ ]:
# §2 — edit ONLY if you used Option B.
import os, subprocess

REPO = '/kaggle/working/bapinns'   # Option B: '/kaggle/input/<your-dataset>'
GIT_URL = ''  # Option A: paste your repo URL, e.g. 'https://github.com/<you>/bapinnsformer.git'

if GIT_URL and not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO], check=True)
assert os.path.isfile(os.path.join(REPO, 'tasks.py')), f'no tasks.py under {REPO}'
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, 'src'))
print('repo:', REPO)
!python tasks.py --list

## §3 — Attach datasets (read this fully before running)

| Need | Kaggle way | Local/laptop way |
|---|---|---|
| **D1 CPCB hourly** (primary) | Attach Kaggle input **“Air Quality Data in India (2015–2020)”** (station-wise hourly CPCB mirror; covers Oct 2019–Dec 2020) | Manual CSV export from https://airquality.cpcb.gov.in/ccr/#/ → `data/raw/cpcb/` |
| **D1 top-up 2021–2024** | Upload CCR CSVs as a **private** Kaggle dataset, attach it | Same `data/raw/cpcb/` folder |
| **D3 ERA5** (wind/BLH) | Upload once-downloaded NetCDFs as a **private** dataset (fetching multi-GB inside sessions is flaky) | CDS API → `data/raw/era5/` |
| **D4 FIRMS fires** (E4 only, held out) | Upload CSV as private dataset **only when you reach §9** | FIRMS API → `data/heldout/firms/` |
| ⚠️ GitHub `cp099/India-Air-Quality-Dataset` | **Do NOT use** — daily city-level *estimated* values, no stations/hourly/met | — |

Wire the paths in the next cell. It fails loudly if anything is missing.

In [ ]:
# §3 — EDIT THESE to your attached input paths, then run.
import glob, os

CPCB_INPUT = '/kaggle/input/air-quality-data-in-india'  # D1b mirror (or '' if using data/raw/cpcb/)
CPCB_EXTRA = ''   # your private CCR 2021-2024 dataset mount, if any
ERA5_INPUT = ''   # your private ERA5 dataset mount, if any

def show(label, pattern, required):
    hits = sorted(glob.glob(pattern, recursive=True))
    print(f'{label}: {len(hits)} files' + (f'  e.g. {hits[0]}' if hits else ''))
    if required and not hits:
        raise FileNotFoundError(f'MISSING {label}: check the §3 table. Pattern: {pattern}')
    return hits

raw_cpcb = show('raw/cpcb', 'data/raw/cpcb/**/*', required=False)
if CPCB_INPUT:
    show('kaggle D1b', f'{CPCB_INPUT}/**/*', required=True)
if ERA5_INPUT:
    show('kaggle D3', f'{ERA5_INPUT}/**/*', required=True)
if not raw_cpcb and not CPCB_INPUT:
    raise FileNotFoundError('No CPCB data: attach the Kaggle D1b dataset or fill data/raw/cpcb/ (see §3 table).')
print('dataset wiring ok')

## §4 — Smoke test (CPU, ~1 min, no data needed)

Must print `task 'smoke' done.` If not, stop and fix before touching data.

In [ ]:
# §4
!python tasks.py smoke

## §5 — E0 data pipeline (CPU)

Needs §3 data. Rebuilds `data/processed/` end-to-end + data-quality report.

In [ ]:
# §5
!python tasks.py data

## §6 — E1 identifiability sweep (CPU; the publishable fallback)

No real data needed — can run before §5 finishes. Full grid takes a while on CPU; start with `-- --quick` for one cell, then the full sweep.

In [ ]:
# §6 — quick cell first (uncomment full run when happy)
!python tasks.py e1 -- --quick
# !python tasks.py e1

## §7 — E2 real fits (GPU, session-capped)

1. Run the **single debug fit** first (validates wiring on your data, minutes).
2. Then fan out. **Two-T4 strategy:** open this notebook in a second session and run a *different* season/pollutant there — never start a fit longer than the session time left.

In [ ]:
# §7a — single debug fit (GPU)
!python scripts/20_fit_real.py --config configs/experiment/e2_real.yaml --city delhi_ncr --season postmonsoon --pollutant PM2.5

In [ ]:
# §7b — fan-out across seasons x pollutants (GPU, long). Cap it per session:
# examples: --max-fits 1  (one instance, safest)  |  default = all instances
!python tasks.py e2 -- --max-fits 1

## §8 — E3 ablations (GPU)

4 pseudo-sequence variants × L at matched params + profiling. Timing-only rows work without trained fits; accuracy rows need §7 checkpoints.

In [ ]:
# §8
!python tasks.py e3

## §9 — E4 fire validation (CPU; held-out discipline!)

Attach the FIRMS CSV **now** (private dataset or `data/heldout/firms/`). It must NEVER enter training — this cell only correlates recovered inflow against it. Verify the guard first.

In [ ]:
# §9
!python -m pytest tests/test_leakage.py -q
# Fill the two paths, then uncomment:
# !python scripts/40_validate_fires.py --config configs/experiment/e4_validation.yaml --inflow-csv results/e2_real/<run>/nw_inflow_daily.csv --frp-csv data/heldout/firms/frp_daily.csv

## §10 — E5 transfer airshed (GPU)

Same pipeline, new domain file. Edit `configs/experiment/e5_transfer.yaml` for Kolkata/Chennai inputs first.

In [ ]:
# §10
!python tasks.py e5

## §11 — Figures, tables, outputs (CPU)

Regenerates `paper/figures/` + `paper/tables/` from `results/`. Then use Kaggle **Save Version → Output** to download `results/` + figures.

In [ ]:
# §11
!python tasks.py figures
import glob
print('figures:', sorted(glob.glob('paper/figures/*.png')))
print('result runs:', sorted(glob.glob('results/runs/*'))[:5])